# 00 — Data Pipeline
**Global Supply Chain Intelligence**

This notebook walks through the complete data ingestion pipeline:
1. FRED API macro indicators (or synthetic fallback)
2. UN Comtrade trade flow data (or synthetic fallback)
3. Synthetic manufacturing data (SKUs, demand, disruption events)
4. DuckDB schema creation and data loading
5. Feature engineering SQL views

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import duckdb
from src.ingest import run_pipeline
from src.synthetic import save_all

/Users/harthikmallichetty/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Run Full Pipeline

In [2]:
# Run the full data pipeline
# NOTE: This rebuilds the DB from scratch. Skip if DB already exists.
from pathlib import Path
db_path = Path('../data/processed/supply_chain.db')

if not db_path.exists():
    all_good = run_pipeline()
else:
    print(f"Database already exists at {db_path} ({db_path.stat().st_size / 1e6:.1f} MB)")
    print("To rebuild, delete the DB file and re-run this cell.")
    all_good = True

Database already exists at ../data/processed/supply_chain.db (2.6 MB)
To rebuild, delete the DB file and re-run this cell.


## Inspect DuckDB Tables

In [3]:
con = duckdb.connect('../data/processed/supply_chain.db', read_only=True)

# Table counts
tables = ['macro_indicators', 'trade_flows', 'skus', 'weekly_demand', 'disruption_events']
for t in tables:
    count = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'{t}: {count:,} rows')

macro_indicators: 840 rows
trade_flows: 840 rows
skus: 500 rows
weekly_demand: 78,000 rows
disruption_events: 3 rows


In [4]:
# Sample macro indicators
con.execute('SELECT * FROM macro_indicators WHERE series_id = 'BDIY' LIMIT 10').fetchdf()

,date,series_id,value,series_name,yoy_change,mom_change
0,2018-01-01,BDIY,1297.3875,Baltic Dry Index,NaN,NaN
1,2018-02-01,BDIY,1741.8598,Baltic Dry Index,NaN,34.259024
2,2018-03-01,BDIY,1690.1340,Baltic Dry Index,NaN,-2.969573
3,2018-04-01,BDIY,1576.9619,Baltic Dry Index,NaN,-6.696043
4,2018-05-01,BDIY,1645.8268,Baltic Dry Index,NaN,4.366935
5,2018-06-01,BDIY,1564.7941,Baltic Dry Index,NaN,-4.923525
6,2018-07-01,BDIY,1395.1731,Baltic Dry Index,NaN,-10.839829
7,2018-08-01,BDIY,1246.5830,Baltic Dry Index,NaN,-10.650299
8,2018-09-01,BDIY,1569.2964,Baltic Dry Index,NaN,25.887839
9,2018-10-01,BDIY,1476.2620,Baltic Dry Index,NaN,-5.928415


In [5]:
# SKU distribution by category
con.execute("""
    SELECT category, COUNT(*) as count, 
           AVG(lead_time_days) as avg_lead_time,
           AVG(unit_cost_usd) as avg_cost
    FROM skus GROUP BY category ORDER BY count DESC
""").fetchdf()

,category,count,avg_lead_time,avg_cost
0,Auto Parts,100,33.250000,168.811600
1,Food & Beverage,80,35.275000,47.566875
2,Electronics,80,37.112500,263.804125
3,Pharmaceuticals,70,35.014286,443.286286
4,Chemicals,60,35.466667,108.658833
5,Metals,60,32.800000,149.571500
6,Textiles,50,35.260000,53.493200


In [6]:
# Disruption events
con.execute('SELECT * FROM disruption_events WHERE severity_score >= 0.7').fetchdf()

,event_id,event_name,start_date,end_date,affected_countries,affected_hs_codes,severity_score
0,EVT001,Ukraine Conflict Onset,2022-02-21,2022-04-18,"Turkey,Germany","1001,1512,2814,2804,7601,7502",0.85
1,EVT002,Red Sea Shipping Disruption,2023-10-02,2023-12-25,"China,India,Vietnam,Thailand,Taiwan,South Kore...","8541,8703,8708,2709,7601,7502",0.75
2,EVT003,Port of Singapore Congestion,2024-05-13,2024-07-08,"China,Vietnam,Thailand,Taiwan,South Korea,Japa...","8541,8703,8708,2804",0.65


In [7]:
# Feature engineering views
print('\n--- Supplier Concentration Index (HHI) ---')
print(con.execute('SELECT * FROM supplier_concentration_index').fetchdf())

print('\n--- Top Disrupted Commodities ---')
print(con.execute("""
    SELECT * FROM commodity_disruption_score 
    WHERE ABS(z_score) > 1.5 ORDER BY ABS(z_score) DESC LIMIT 10
""").fetchdf())


--- Supplier Concentration Index (HHI) ---


,category,hhi_index,concentration_level,num_supplier_countries
0,Metals,1413.135160,Low Concentration,10
1,Chemicals,1876.245833,Moderate Concentration,11
2,Pharmaceuticals,1901.874611,Moderate Concentration,11
3,Food & Beverage,1814.711147,Moderate Concentration,11
4,Electronics,1761.569039,Moderate Concentration,9
5,Auto Parts,1918.720924,Moderate Concentration,11
6,Textiles,1990.249655,Moderate Concentration,9



--- Top Disrupted Commodities ---


,year,reporter_code,reporter_name,hs_code,commodity_name,flow_type,trade_value_usd,baseline_mean,baseline_std,z_score,disruption_flag
0,2022,392,Japan,2814,"Ammonia, anhydrous or solution",Export,2.598192e+08,1.977677e+08,8.902997e+06,6.969730,True
1,2024,699,India,7502,Unwrought nickel,Export,5.254000e+08,3.965964e+08,1.946139e+07,6.618418,True
2,2023,842,USA,2709,"Petroleum oils, crude",Export,1.793432e+10,2.365412e+10,8.768939e+08,-6.522792,True
3,2022,410,South Korea,7502,Unwrought nickel,Import,4.220211e+08,5.833263e+08,2.474406e+07,-6.518947,True
4,2024,410,South Korea,7502,Unwrought nickel,Import,7.434906e+08,5.833263e+08,2.474406e+07,6.472840,True
5,2022,699,India,2814,"Ammonia, anhydrous or solution",Export,2.096825e+08,1.289011e+08,1.255597e+07,6.433707,True
6,2024,410,South Korea,8708,Parts for motor vehicles,Import,3.013053e+09,4.659954e+09,2.597996e+08,-6.339120,True
7,2023,699,India,7502,Unwrought nickel,Export,5.126679e+08,3.965964e+08,1.946139e+07,5.964191,True
8,2024,156,China,2814,"Ammonia, anhydrous or solution",Export,5.364922e+08,3.723451e+08,2.859455e+07,5.740501,True
9,2022,699,India,8708,Parts for motor vehicles,Import,4.567220e+09,3.078932e+09,2.608513e+08,5.705504,True


In [8]:
con.close()